In [7]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [8]:
# Inception Block Architecture

# - Branch 1 ~ 3: 각각 다른 window size를 사용해 spatial scale 추출
# - Branch 2 ~ 3: 먼저 1x1 Conv를 적용한 다음에 feature를 추출 -> 모델 복잡도 낮아짐.
class Inception(nn.Module):
    
    def __init__(
        self,
        c1: int,
        c2: tuple[int, int],
        c3: tuple[int, int],
        c4: int,
    ) -> None:
        super().__init__()
        
        # Branch 1:
        # 1x1 Convolution Kernel
        self.b1_1 = nn.LazyConv2d(
            out_channels=c1,
            kernel_size=1,
        )
        
        # Branch 2:
        # 1x1 Convolution: Channel 축소
        # 3x3 Convolution: 중간 scale의 feature 추출
        self.b2_1 = nn.LazyConv2d(
            out_channels=c2[0],
            kernel_size=1,
        )
        self.b2_2 = nn.LazyConv2d(
            out_channels=c2[1],
            kernel_size=3,
            padding=1,
        )
        
        # Branch 3:
        # 1x1 Convolution: Channel 축소
        # 5x5 Convolution: 큰 scale의 feature 추출
        self.b3_1 = nn.LazyConv2d(
            out_channels=c3[0],
            kernel_size=1,
        )
        self.b3_2 = nn.LazyConv2d(
            out_channels=c3[1],
            kernel_size=5,
            padding=2,
        )
        
        # Branch 4:
        # MaxPooling 후, 1x1 Convolution으로 Channel 변경
        self.b4_1 = nn.MaxPool2d(
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.b4_2 = nn.LazyConv2d(
            out_channels=c4,
            kernel_size=1,
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        
        # X -> b1_1 -> relu
        branch1 = F.relu(
            self.b1_1(X)
        )
        
        # X -> b2_1 -> relu -> b2_2 -> relu
        branch2 = F.relu(
            self.b2_2(
                F.relu(
                    self.b2_1(X)                    
                )
            ) 
        ) 
        
        # X -> b3_1 -> relu -> b3_2 -> relu
        branch3 = F.relu(
            self.b3_2(
                F.relu(
                    self.b3_1(X)                    
                )
            ) 
        ) 
        
        # X -> b4_1 -> b4_2 -> relu
        branch4 = F.relu(
            self.b4_2(
                self.b4_1(X)
            )
        )
        
        # out_channel: c1 + c2[1] + c3[1] + c4
        # feature map의 크기는 모두 동일하므로 Channel axis를 따라 concatenate 가능
        return torch.cat(
            (
                branch1,
                branch2,
                branch3,
                branch4,
            ),
            dim=1,
        )
        

In [9]:
# Tracing Inception Block Tensor Shape

block = Inception(
    c1=64,
    c2=(96, 128),
    c3=(16, 32),
    c4=32,
)

# [B, C, H, W]
X = torch.randn(
    1,  
    192,
    28,
    28,
)

with torch.no_grad():
    
    # Branch 1
    branch1 = F.relu(
        block.b1_1(X)
    )
    
    # Branch 2
    branch2_reduced = F.relu(
        block.b2_1(X)
    )
    branch2 = F.relu(
        block.b2_2(
            branch2_reduced
        )
    )
    
    # Branch 3
    branch3_reduced = F.relu(
        block.b3_1(X)
    )
    branch3 = F.relu(
        block.b3_2(
            branch3_reduced
        )
    )
    
    # Branch 4
    branch4_pooled = block.b4_1(X)
    branch4 = F.relu(
        block.b4_2(
            branch4_pooled
        )
    )
    
    Y = block(X)
    
    
branch_outputs = (
    ("Input", X),
    ("Branch 1", branch1),
    ("Branch 2 reduction", branch2_reduced),
    ("Branch 2", branch2),
    ("Branch 3 reduction", branch3_reduced),
    ("Branch 3", branch3),
    ("Branch 4 pooling", branch4_pooled),
    ("Branch 4", branch4),
    ("Concatenated output", Y),
)

for name, tensor in branch_outputs:
    print(
        f"{name:<24}",
        tuple(tensor.shape),
    )

Input                    (1, 192, 28, 28)
Branch 1                 (1, 64, 28, 28)
Branch 2 reduction       (1, 96, 28, 28)
Branch 2                 (1, 128, 28, 28)
Branch 3 reduction       (1, 16, 28, 28)
Branch 3                 (1, 32, 28, 28)
Branch 4 pooling         (1, 192, 28, 28)
Branch 4                 (1, 32, 28, 28)
Concatenated output      (1, 256, 28, 28)
